In [0]:
import pyspark.sql.functions as sf

SCHEMA = dbutils.widgets.get("SCHEMA")

TABLE_SILVER_PRODUCTS_RECOMMENDATIONS = SCHEMA + "." + dbutils.widgets.get("TABLE_SILVER_PRODUCTS_RECOMMENDATIONS")
TABLE_GOLD_PRODUCTS_RECOMMENDATIONS = SCHEMA + "." + dbutils.widgets.get("TABLE_GOLD_PRODUCTS_RECOMMENDATIONS") 

In [0]:
df_silver_products_recommendations = spark.read.table(TABLE_SILVER_PRODUCTS_RECOMMENDATIONS)

In [0]:
product_id = (
    df_silver_products_recommendations
    .withColumn("rating", sf.col("rating").try_cast("double"))
    .groupBy(sf.col("product_id"))
    .agg(
        sf.count(sf.col("rating")).alias("cnt"),
        sf.avg(sf.col("rating")).alias("rating_avg"),
        sf.min(sf.col("rating")).alias("rating_min"),
        sf.max(sf.col("rating")).alias("rating_max"),
    )
    .filter(
        (sf.col("rating_min") == 1.0)
        & (sf.col("rating_max") == 5.0)
    )
    .withColumn(
        "diff",
        sf.abs(
            sf.lit(2.5)
            - sf.col("rating_avg")
        )
    )
    .orderBy(sf.col("diff"))
    .filter(sf.col("cnt").between(30, 50))
    .select("product_id")
).collect()[0][0]

In [0]:
df_gold_products_recommendations = (
    df_silver_products_recommendations
    .where(sf.col("product_id") == product_id)
)

(
    df_gold_products_recommendations
    .write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABLE_GOLD_PRODUCTS_RECOMMENDATIONS)
)

spark.sql(f"""
    ALTER TABLE {TABLE_GOLD_PRODUCTS_RECOMMENDATIONS}
    SET TBLPROPERTIES (delta.enableChangeDataFeed = true)
""")